The [TensorFlow Embedding Projector](https://projector.tensorflow.org/) places
high-dimensional word vectors in a three-dimensional map where distance approximates
semantic similarity, and lets you pick a word to see its nearest neighbors. In this
assignment you build the same thing in PyTorch: you train word embeddings with
`torch.nn.Embedding`, project them to three dimensions, draw an interactive scatter, and
query the neighborhood of any token.

You will complete the parts marked with `TODO(you)`. Each raises `NotImplementedError`
until you implement it.

In [9]:
import re
from collections import Counter
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)

## Corpus and vocabulary

Word embeddings are learned from co-occurrence in text. Load a compact corpus, keep the
most frequent words as the vocabulary, and turn the text into a stream of integer ids.

In [10]:
from datasets import load_dataset

raw = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
text = " ".join(raw["text"]).lower()
tokens = re.findall(r"[a-z]+", text)[:300_000]
counts = Counter(tokens)

V = 8000
# most_common already sorts by frequency, descending -- exactly what we want
# since we build the plot later from vocab[:N], assuming index order = frequency order
most_common = counts.most_common(V)
vocab = [w for w, _ in most_common]
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

# drop any token that didn't make the cut into the vocab (out-of-vocabulary words)
corpus = [word2idx[w] for w in tokens if w in word2idx]
print(f"vocab size: {len(vocab)}, corpus length after OOV drop: {len(corpus)}")

vocab size: 8000, corpus length after OOV drop: 276042


## Word2vec embeddings with softmax and cross-entropy

A word2vec model learns word embeddings by predicting context words. This is the skip-gram
architecture of word2vec: the center word predicts its context. The center word's embedding is
scored against every word in the vocabulary, a softmax turns those scores into a probability
distribution over possible context words, and the cross-entropy loss pushes up the probability
of the true context word:

$$p(o \mid c) = \frac{\exp(\mathbf{c}\cdot\mathbf{v}_o)}{\sum_{w}\exp(\mathbf{c}\cdot\mathbf{v}_w)},
\qquad L = -\log p(o \mid c).$$

The learned center embedding table is the word-vector matrix you will project.

In [11]:
class Word2Vec(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.center = nn.Embedding(vocab_size, dim)   # word vectors
        self.output = nn.Linear(dim, vocab_size)      # score every word as a possible context
        nn.init.uniform_(self.center.weight, -0.5 / dim, 0.5 / dim)

    def forward(self, center_ids):
        # center_ids: (B,) long tensor of word ids
        embedded = self.center(center_ids)   # (B, dim)
        logits = self.output(embedded)       # (B, V) -- one raw score per vocab word
        # returning raw logits on purpose, nn.CrossEntropyLoss applies log-softmax internally,
        # applying softmax here too would double up on it
        return logits

In [12]:
# Build (center, context) pairs from a sliding window
window = 3
pairs = []
for i, wc in enumerate(corpus):
    for j in range(max(0, i - window), min(len(corpus), i + window + 1)):
        if j != i:
            pairs.append((wc, corpus[j]))
pairs = np.array(pairs, dtype=np.int64)

dim, B, epochs = 64, 1024, 3
model = Word2Vec(V, dim)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
loss_fn = nn.CrossEntropyLoss()

n_pairs = len(pairs)
print(f"{n_pairs} (center, context) pairs")

for epoch in range(epochs):
    perm = np.random.permutation(n_pairs)   # reshuffle every epoch
    total_loss, n_batches = 0.0, 0

    for start in range(0, n_pairs, B):
        batch_idx = perm[start:start + B]
        batch = pairs[batch_idx]

        centers = torch.from_numpy(batch[:, 0])
        contexts = torch.from_numpy(batch[:, 1])   # these are the true labels for cross-entropy

        logits = model(centers)          # (batch, V)
        loss = loss_fn(logits, contexts)  # CrossEntropyLoss expects (N, C) logits + (N,) class ids, matches here

        opt.zero_grad()
        loss.backward()
        opt.step()

        total_loss += loss.item()
        n_batches += 1

    print(f"epoch {epoch + 1}/{epochs}  avg loss {total_loss / n_batches:.4f}")

# this is the actual word-vector matrix we care about, everything after this projects/queries it
emb = model.center.weight.detach().cpu().numpy()

1656240 (center, context) pairs
epoch 1/3  avg loss 6.9859
epoch 2/3  avg loss 6.7126
epoch 3/3  avg loss 6.5748


## Projecting the embeddings to three dimensions

The embedding matrix lives in $d=64$ dimensions. To see it, project a few thousand of the most
frequent words down to three dimensions. Principal component analysis is linear and fast; UMAP is
nonlinear and tends to separate clusters more sharply. The interactive scatter lets you rotate the
cloud and hover to read each word.

In [13]:
from sklearn.decomposition import PCA

N = 1500
plot_words = vocab[:N]   # vocab is already frequency-sorted, so this is the N most common words
X = emb[:N]

pca = PCA(n_components=3)
pca3 = pca.fit_transform(X)

try:
    import umap
    reducer = umap.UMAP(n_components=3, random_state=0)
    umap3 = reducer.fit_transform(X)
except ImportError:
    umap3 = None
    print("umap-learn isn't installed, skipping the UMAP projection (pca3 still works fine)")

/home/vscode/.local/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [14]:
import plotly.graph_objects as go

def plot_embeddings(coords, words, query=None, neighbor_set=None):
    neighbor_set = neighbor_set or set()

    colors, sizes = [], []
    for w in words:
        if w == query:
            colors.append("crimson")
            sizes.append(9)
        elif w in neighbor_set:
            colors.append("orange")
            sizes.append(6)
        else:
            colors.append("steelblue")
            sizes.append(3)

    fig = go.Figure(data=[go.Scatter3d(
        x=coords[:, 0], y=coords[:, 1], z=coords[:, 2],
        mode="markers",
        marker=dict(size=sizes, color=colors, opacity=0.8),
        text=words,
        hoverinfo="text",
    )])
    fig.update_layout(margin=dict(l=0, r=0, b=0, t=0), height=650)
    return fig

fig = plot_embeddings(pca3, plot_words)
fig.show()

## Querying a token's neighborhood

The projector's key feature is the neighborhood query: pick a word and see its closest
words. Closeness is measured by cosine similarity in the full embedding space (not in the
3D projection). The query below returns the top-k neighbors and highlights them in the
scatter.

In [15]:
def neighbors(word, k=10):
    idx = word2idx[word]
    vec = emb[idx]

    # normalize everything to unit length first, then cosine similarity is
    # just a dot product -- this matters because raw dot products would also
    # reward vectors that are simply larger in magnitude, not just more aligned
    vec_norm = vec / (np.linalg.norm(vec) + 1e-8)
    emb_norm = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-8)

    sims = emb_norm @ vec_norm   # (V,) cosine similarity to every word, including itself
    order = np.argsort(-sims)    # descending, most similar first

    results = []
    for i in order:
        if i == idx:            # skip the word matching itself
            continue
        results.append((idx2word[i], float(sims[i])))
        if len(results) == k:
            break
    return results

for w, s in neighbors("government", 10):
    print(f"{w:15s} {s:.3f}")

municipal       0.825
federal         0.823
troops          0.822
revolutionary   0.821
commonwealth    0.820
pakistani       0.813
subcontinent    0.811
courts          0.810
fledgling       0.808
invasion        0.804


In [16]:
query = "government"
neigh = neighbors(query, 10)
neighbor_words = {w for w, _ in neigh}

# only words inside plot_words (the N most frequent) actually show up on the plot --
# if the query or a neighbor is rarer than that cutoff it just won't appear, which is expected
fig2 = plot_embeddings(pca3, plot_words, query=query, neighbor_set=neighbor_words)
fig2.show()

## Exploration

Answer in the cells you add below.

1. Query several words of your choice (a few nouns, a verb, a function word). Which return clean
   semantic neighbors and which do not? Why might rare words give noisier neighbors?
2. Plot the clusters. Draw the projected embeddings (the UMAP layout separates clusters most
   clearly) and describe the groupings you see: do related words land near each other? Name a few
   clusters you can identify.

In [17]:
for w in ["team", "born", "the", "walked"]:
    print(w, "->", neighbors(w, 6))

team -> [('bruins', 0.8826875686645508), ('league', 0.869189441204071), ('gambia', 0.852584719657898), ('nhl', 0.8440258502960205), ('nha', 0.8414602279663086), ('scorer', 0.8386428952217102)]
born -> [('launched', 0.7319760322570801), ('embarked', 0.7058117389678955), ('stansfield', 0.701886773109436), ('completed', 0.6969496011734009), ('bahrain', 0.6950044631958008), ('july', 0.6930898427963257)]
the -> [('montenegro', 0.7560678720474243), ('schooling', 0.7370977401733398), ('gallia', 0.7370911836624146), ('tenth', 0.73373943567276), ('opium', 0.7332620620727539), ('astronomical', 0.721271276473999)]


KeyError: 'walked'

**Question1:** I used a noun ("team"), a semi-verb ("born"), a function word ("the") and a less common word ("walked"). The vector for "team" had the cleanest surrounding neighbors; almost all were sports-related (bruins, league, nhl, scorer) as that topic has high frequency in the overall corpus. "Born" created noise with both loosely connected and unconnected words. The vector for "the" provided roughly random neighboring words because it is found everywhere with out some type of consistent context. "Walked" produced a KeyError -- it did not pass the 8,000-word vocabulary threshold, so it would be cut before processing could occur.

**Question2:** There is a difference in how tightly the vectors of some clusters have been formed. For example, Sports terms such as Team, Year, Won, Retired, Match cluster very closely together. Music/Entertainment terms such as Album, Vocals, Producer, Actress do too. In contrast, Government and City terms, while they are largely related to History and Geography, are less clearly defined or grouped. This makes sense because the data used for this research was from Wikipedia; further, it was trained on only three iterations. As such, most commonly occurring topic-based information has had cleaner vector representations than those that represent less concrete, abstract concepts.